# EDA — Sleep Health and Lifestyle Dataset
### Hiểu dữ liệu trước khi mô hình hoá

Mỗi phần gồm ba bước: khái niệm ngắn gọn, code, và cách đọc kết quả.


In [ ]:

import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib as mpl
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
import warnings; warnings.filterwarnings('ignore')

mpl.rcParams.update({'figure.dpi':110,'savefig.dpi':150,'savefig.bbox':'tight',
    'font.size':11,'axes.titlesize':12.5,'axes.titleweight':'bold','axes.labelsize':11,
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,
    'grid.color':'#e5e7eb','figure.facecolor':'white','legend.frameon':False})
sns.set_palette('Set2')
CLASS_COLORS = {'None':'#2563eb','Sleep Apnea':'#ef4444','Insomnia':'#f59e0b'}
os.makedirs('figs', exist_ok=True)
print('Thư viện sẵn sàng.')


## 1. Tổng quan dữ liệu

Bước đầu là làm quen với dữ liệu: số dòng, số cột, kiểu dữ liệu, và có giá trị thiếu hay không.


In [ ]:

df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')   # NaN nghĩa là không bị rối loạn
df[['Systolic','Diastolic']] = df['Blood Pressure'].str.split('/', expand=True).astype(int)
df = df.drop(columns=['Person ID','Blood Pressure'])

print('Kích thước (dòng, cột):', df.shape)
print('\nSố giá trị thiếu mỗi cột:'); print(df.isnull().sum().to_string())
print('\nKiểu dữ liệu:'); print(df.dtypes.to_string())
df.head()


In [ ]:

df.describe().round(2)


## 2. Mất cân bằng lớp

Mất cân bằng lớp xảy ra khi các lớp có số mẫu chênh lệch lớn. Ở đây lớp None chiếm đa số.

Vấn đề: mô hình có thể chỉ đoán lớp đa số mà vẫn đạt accuracy cao, nhưng bỏ qua lớp thiểu số.
Giải pháp: dùng macro precision, recall, F1 và kỹ thuật cân bằng như SMOTE.


In [ ]:

cnt = df['Sleep Disorder'].value_counts()
tab = pd.DataFrame({'Số mẫu': cnt, 'Tỷ lệ (%)': (cnt/len(df)*100).round(2)})
print(tab.to_string())
print(f"\nMức mất cân bằng (lớp lớn nhất / nhỏ nhất): {cnt.max()/cnt.min():.2f} : 1")

fig, ax = plt.subplots(figsize=(6,4))
order = ['None','Sleep Apnea','Insomnia']
bars = ax.bar(order, [cnt[c] for c in order], color=[CLASS_COLORS[c] for c in order], alpha=.9)
for b,c in zip(bars, order):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+3, f"{cnt[c]}\n{cnt[c]/len(df)*100:.1f}%",
            ha='center', fontsize=10)
ax.set_ylabel('Số bệnh nhân'); ax.set_title('Phân bố lớp — dữ liệu gốc')
ax.set_ylim(0, cnt.max()*1.2)
plt.savefig('figs/01_class_imbalance.png'); plt.show()


## 3. SMOTE — cân bằng dữ liệu

SMOTE tạo thêm mẫu tổng hợp cho lớp thiểu số bằng cách nội suy giữa các mẫu gần nhau, thay vì sao chép.
Sau khi áp dụng, các lớp trở nên cân bằng.

Lưu ý: trong huấn luyện thật, SMOTE chỉ áp dụng trên tập train để tránh rò rỉ dữ liệu sang tập test.
Ở đây SMOTE chạy trên toàn bộ chỉ để minh hoạ khái niệm.


In [ ]:

X_demo = df.drop(columns=['Sleep Disorder']).copy()
for c in X_demo.select_dtypes('object').columns:
    X_demo[c] = LabelEncoder().fit_transform(X_demo[c])
y_demo = df['Sleep Disorder']

X_res, y_res = SMOTE(random_state=42).fit_resample(X_demo, y_demo)

before = y_demo.value_counts(); after = pd.Series(y_res).value_counts()
print(pd.DataFrame({'Trước SMOTE': before, 'Sau SMOTE': after}).to_string())

fig, ax = plt.subplots(1,2, figsize=(11,4), sharey=True)
for a,(title,s) in zip(ax, [('Trước SMOTE', before),('Sau SMOTE', after)]):
    a.bar(order, [s[c] for c in order], color=[CLASS_COLORS[c] for c in order], alpha=.9)
    a.set_title(title); a.set_ylabel('Số mẫu')
    for i,c in enumerate(order): a.text(i, s[c]+3, str(s[c]), ha='center')
fig.suptitle('SMOTE đưa các lớp thiểu số lên bằng lớp đa số', fontweight='bold', y=1.03)
plt.savefig('figs/02_smote_before_after.png'); plt.show()


## 4. Phân phối đặc trưng số — Histogram

Histogram chia trục giá trị thành các khoảng và đếm số mẫu trong mỗi khoảng.
Nó cho thấy hình dạng phân phối: lệch trái hay phải, một đỉnh hay nhiều đỉnh, có giá trị bất thường không.


In [ ]:

num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color='#2563eb', alpha=.5, edgecolor='white')
    ax.set_title(col); ax.set_xlabel(''); ax.set_ylabel('')
for ax in axes.ravel()[len(num_cols):]: ax.set_visible(False)
fig.suptitle('Phân phối của các biến số', fontweight='bold')
plt.tight_layout(); plt.savefig('figs/03_histograms.png'); plt.show()


Khi tách histogram theo lớp bệnh, biến nào có phân phối lệch nhau rõ giữa các lớp thì biến đó
có khả năng phân biệt lớp tốt.

In [ ]:

key_feats = ['Systolic','Diastolic','Sleep Duration','Heart Rate','Daily Steps','Age']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), key_feats):
    for cls in order:
        sns.kdeplot(df[df['Sleep Disorder']==cls][col], ax=ax, fill=True, alpha=.25,
                    color=CLASS_COLORS[cls], label=cls, linewidth=1.5)
    ax.set_title(col); ax.set_ylabel('')
axes[0,0].legend(fontsize=8)
fig.suptitle('Mật độ từng biến theo lớp', fontweight='bold')
plt.tight_layout(); plt.savefig('figs/04_dist_by_class.png'); plt.show()


## 5. Ma trận tương quan

Tương quan Pearson đo mức độ hai biến số đi cùng nhau theo đường thẳng, giá trị từ -1 đến 1.
Gần 1 là cùng tăng, gần -1 là ngược chiều, gần 0 là không liên hệ tuyến tính.

Hai biến tương quan rất mạnh nghĩa là chúng trùng thông tin. Đây là một lý do để giảm chiều dữ liệu.


In [ ]:

corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, cbar_kws={'shrink':.8}, ax=ax, annot_kws={'size':8})
ax.set_title('Ma trận tương quan Pearson giữa các biến số')
plt.savefig('figs/05_correlation_heatmap.png'); plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape),1).astype(bool)).stack().reset_index())
pairs.columns=['Biến 1','Biến 2','r']
print('Các cặp tương quan mạnh, |r| > 0.5:')
print(pairs[pairs.r.abs()>0.5].sort_values('r', key=abs, ascending=False).to_string(index=False))


## 6. Biến phân loại theo lớp

Với biến phân loại, count plot đếm số mẫu mỗi nhóm và tách theo lớp bệnh.
Nếu tỷ lệ lớp bệnh khác nhau giữa các nhóm thì biến đó mang thông tin có ích.


In [ ]:

cat_cols = ['Gender','BMI Category','Occupation']
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, cat_cols):
    sns.countplot(data=df, y=col, hue='Sleep Disorder', ax=ax,
                  palette=CLASS_COLORS, hue_order=order, order=df[col].value_counts().index)
    ax.set_title(col); ax.set_xlabel('Số mẫu'); ax.set_ylabel('')
    if ax.legend_: ax.legend_.remove()
axes[0].legend(title='Sleep Disorder', fontsize=8)
fig.suptitle('Phân bố biến phân loại theo lớp bệnh', fontweight='bold')
plt.tight_layout(); plt.savefig('figs/06_categorical.png'); plt.show()


## 7. Boxplot

Boxplot tóm tắt phân phối bằng năm số: nhỏ nhất, Q1, trung vị, Q3, lớn nhất.
Hộp là khoảng giữa Q1 và Q3, điểm rời rạc là giá trị bất thường.
Đặt cạnh nhau theo lớp giúp thấy biến nào dịch chuyển giữa các lớp.


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for ax, col in zip(axes.ravel(), key_feats):
    sns.boxplot(data=df, x='Sleep Disorder', y=col, order=order, ax=ax,
                palette=CLASS_COLORS, width=.6)
    ax.set_title(col); ax.set_xlabel(''); ax.tick_params(axis='x', labelsize=8)
fig.suptitle('Boxplot từng biến theo lớp', fontweight='bold')
plt.tight_layout(); plt.savefig('figs/07_boxplots.png'); plt.show()


## 8. Chiếu dữ liệu xuống 2D: PCA, LDA, UMAP

Dữ liệu nhiều chiều không vẽ trực tiếp được, nên chiếu xuống 2D để nhìn cấu trúc lớp. Ba cách:

- PCA: tuyến tính, không dùng nhãn, giữ phương sai lớn nhất.
- LDA: tuyến tính, dùng nhãn, tách các lớp xa nhau nhất.
- UMAP: phi tuyến, giữ cấu trúc hàng xóm cục bộ.

Cách đọc: nếu PCA hoặc LDA đã tách lớp gọn thì dữ liệu tách được bằng đường thẳng.
Nếu chỉ UMAP tách được còn PCA và LDA rối thì cấu trúc là phi tuyến.


In [ ]:

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import umap

Xnum = StandardScaler().fit_transform(df[num_cols])   # chuẩn hoá trước khi chiếu
y = df['Sleep Disorder'].values

pca = PCA(n_components=2).fit_transform(Xnum)
lda = LDA(n_components=2).fit_transform(Xnum, y)
um  = umap.UMAP(n_components=2, random_state=42).fit_transform(Xnum)

def scatter(ax, emb, title, sub):
    for cls in order:
        m = y==cls
        ax.scatter(emb[m,0], emb[m,1], s=22, alpha=.7, color=CLASS_COLORS[cls], label=cls,
                   edgecolors='white', linewidths=.3)
    ax.set_title(title); ax.set_xlabel(sub+' 1'); ax.set_ylabel(sub+' 2')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
scatter(axes[0], pca, 'PCA — tuyến tính, không giám sát', 'PC')
scatter(axes[1], lda, 'LDA — tuyến tính, có giám sát', 'LD')
scatter(axes[2], um,  'UMAP — phi tuyến', 'UMAP')
axes[0].legend(fontsize=8)
fig.suptitle('Chiếu 2D — so sánh khả năng tách lớp của ba phương pháp', fontweight='bold')
plt.tight_layout(); plt.savefig('figs/08_projections.png'); plt.show()


So sánh mức độ tách lớp của LDA và UMAP. Nếu LDA tách tốt gần bằng UMAP thì dữ liệu này
về cơ bản tách được tuyến tính, nên không cần đến phương pháp phi tuyến.

## 9. Tổng kết

1. Dữ liệu mất cân bằng, nên dùng macro-metric và SMOTE, không tin accuracy đơn lẻ.
2. SMOTE tạo mẫu tổng hợp cho lớp thiểu số, và chỉ áp dụng trên tập train.
3. Histogram, KDE và boxplot cho thấy biến nào tách lớp tốt.
4. Heatmap tương quan chỉ ra biến trùng thông tin, là động lực để giảm chiều.
5. PCA, LDA, UMAP giúp trả lời câu hỏi dữ liệu tuyến tính hay phi tuyến.

Tất cả hình được lưu trong thư mục figs.
